# 1 · Prepare data (in-notebook) + inspect

Builds the disjoint TL;DR splits locally (CPU) and shows a sample SFT row and a GRPO row, then uploads to S3. Data prep is cheap — no training job burned on it.

In [ ]:
import json, sys, os
sys.path.insert(0, '../src')
cfg = json.load(open('summ_config.json'))
os.environ['HF_TOKEN'] = cfg['hf_token']
from transformers import AutoTokenizer
import data_utils

tok = AutoTokenizer.from_pretrained(cfg['base_model'], trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

eval_rows, grpo_rows, sft_rows = data_utils.load_and_split(
    cfg['tldr_dataset'], tok, n_eval=cfg['n_eval'], n_grpo=cfg['n_grpo'],
    n_train=cfg['n_train'], seed=cfg['seed'], enable_thinking=bool(cfg['enable_thinking']))
print('eval/grpo/sft:', len(eval_rows), len(grpo_rows), len(sft_rows))

In [ ]:
# Peek: an SFT row (prompt + reference completion) and a GRPO row (prompt + source post).
print('=== SFT reference (completion) ===')
print(sft_rows[0]['completion'][:400])
print('\n=== GRPO document (post the judge will score against) ===')
print(grpo_rows[0]['document'][:400])

In [ ]:
# Write splits locally, then upload to S3.
import data_utils, common
n_val = max(1, int(0.02 * len(sft_rows)))
sft_val, sft_train = sft_rows[:n_val], sft_rows[n_val:]
paths = common.local_paths('./job_out')
data_utils.write_jsonl(paths['data'] + '/sft_train.jsonl', sft_train, ['prompt', 'completion'])
data_utils.write_jsonl(paths['data'] + '/sft_val.jsonl', sft_val, ['prompt', 'completion'])
data_utils.write_jsonl(paths['data'] + '/grpo_train.jsonl', grpo_rows, ['prompt', 'document', 'reference'])
data_utils.write_jsonl(paths['data'] + '/eval.jsonl', eval_rows, ['prompt', 'document', 'reference'])
common.s3_upload_dir(paths['data'], cfg['data_s3'])
print('uploaded ->', cfg['data_s3'])

**Next:** `02_sft.ipynb`.